In [ ]:
import pandas as pd
from pypfopt import EfficientFrontier, risk_models, expected_returns

# Chargement des données historiques
data = pd.read_csv(
    "df_historiques.csv",
    index_col="Date",
    parse_dates=True
)

# Sélection des 23 actifs ayant les rendements moyens les plus élevés
returns = data.pct_change().dropna()
top_assets = returns.mean().nlargest(23).index

data_subset = data[top_assets]

# Estimation des rendements espérés et de la covariance
mu = expected_returns.mean_historical_return(data_subset)
S = risk_models.CovarianceShrinkage(data_subset).ledoit_wolf()

# Construction du portefeuille optimisé
risk_free_rate = 0.04

ef = EfficientFrontier(mu, S)
ef.add_constraint(lambda w: w <= 0.2)

weights = ef.max_sharpe(risk_free_rate=risk_free_rate)

# Nettoyage et affichage des poids optimaux
cleaned_weights = ef.clean_weights()

weights_df = pd.DataFrame(
    cleaned_weights.items(),
    columns=["Asset", "Weight"]
)

print("Optimal Portfolio Weights:")
print(weights_df)

# Évaluation de la performance du portefeuille
performance = ef.portfolio_performance(
    risk_free_rate=risk_free_rate,
    verbose=True
)

# Exportation des poids optimaux
weights_df.to_csv(
    "optimal_portfolio_weights.csv",
    index=False
)


Poids optimaux pour les actions sélectionnées :
OrderedDict({'TSLA': np.float64(0.12585), 'NVDA': np.float64(0.23763), 'NFLX': np.float64(0.08776), 'UNH': np.float64(0.49577), 'MSFT': np.float64(0.02662), 'ADBE': np.float64(0.0), 'AAPL': np.float64(0.0), 'AMZN': np.float64(0.0), 'MA': np.float64(0.0), 'HD': np.float64(0.0), 'V': np.float64(0.0), 'GOOGL': np.float64(0.0), 'JPM': np.float64(0.0), 'BAC': np.float64(0.0), 'GS': np.float64(0.0), 'PEP': np.float64(0.0), 'JNJ': np.float64(0.02637), 'PFE': np.float64(0.0), 'PG': np.float64(0.0), 'CMCSA': np.float64(0.0), 'KO': np.float64(0.0), 'INTC': np.float64(0.0), 'DIS': np.float64(0.0)})
Expected annual return: 35.4%
Annual volatility: 25.3%
Sharpe Ratio: 1.32
